In [9]:
from collections import deque
import pandas as pd
import os

In [10]:
project_dir = os.path.dirname(os.getcwd())
games = pd.read_csv(os.path.join(project_dir, "data/cleaned_chess.csv"))

In [11]:
games

,Event,Site,Date,White,Black,Result,WhiteElo,BlackElo,ECO,Opening
0,1st CT Casino Marina del Sol,Talcahuano CHI,2012-06-02,"Caceres Leal,Jaime","Farfan Ortiz,V",0.5,2137.0,2206.0,D13,QGD Slav
1,1st CT Casino Marina del Sol,Talcahuano CHI,2012-06-02,"Caceres Leal,Jaime","Salinas Herrera,P",0.0,2137.0,2409.0,A46,Queen's pawn game
2,1st CT Casino Marina del Sol,Talcahuano CHI,2012-06-02,"Farfan Ortiz,E","Vizama Riveros,C",0.0,2236.0,2256.0,C42,Petrov
3,1st CT Casino Marina del Sol,Talcahuano CHI,2012-06-02,"Rojas Sepulveda,E","Diaz Vasquez,M",0.5,2315.0,2212.0,A46,Queen's pawn
4,1st CT Casino Marina del Sol,Talcahuano CHI,2012-06-02,"Estay Torreblanca,O","Alarcon,Rod",1.0,2214.0,2174.0,C17,French
...,...,...,...,...,...,...,...,...,...,...
3255687,Vladimir Dvorkovich Open,Aktobe KAZ,2026-06-08,"Kabinazar,Nurmukhammed","Tarhan,Adar",0.5,2312.0,2482.0,B11,Caro-Kann
3255688,Vladimir Dvorkovich Open,Aktobe KAZ,2026-06-08,"Zhalmakhanov,Ramazan","Kornienko,Konstantin",0.0,2478.0,2388.0,B28,Sicilian
3255689,Vladimir Dvorkovich Open,Aktobe KAZ,2026-06-08,"Dronavalli,Harika","Degenbaev,Aziz",0.5,2466.0,2320.0,A06,Reti opening
3255690,Vladimir Dvorkovich Open,Aktobe KAZ,2026-06-08,"Rozum,I","Makhnev,Denis",1.0,2455.0,2551.0,A40,Queen's pawn


Elo 

In [12]:
games["EloDiff"] = games["WhiteElo"] - games["BlackElo"]
games["AvgElo"] = (games["WhiteElo"] + games["BlackElo"]) / 2
games["HigherRatedPlayer"] = games.loc[:,["WhiteElo", "BlackElo"]].idxmax(axis="columns").replace("BlackElo", 0).replace("WhiteElo", 1)
games

,Event,Site,Date,White,Black,Result,WhiteElo,BlackElo,ECO,Opening,EloDiff,AvgElo,HigherRatedPlayer
0,1st CT Casino Marina del Sol,Talcahuano CHI,2012-06-02,"Caceres Leal,Jaime","Farfan Ortiz,V",0.5,2137.0,2206.0,D13,QGD Slav,-69.0,2171.5,0
1,1st CT Casino Marina del Sol,Talcahuano CHI,2012-06-02,"Caceres Leal,Jaime","Salinas Herrera,P",0.0,2137.0,2409.0,A46,Queen's pawn game,-272.0,2273.0,0
2,1st CT Casino Marina del Sol,Talcahuano CHI,2012-06-02,"Farfan Ortiz,E","Vizama Riveros,C",0.0,2236.0,2256.0,C42,Petrov,-20.0,2246.0,0
3,1st CT Casino Marina del Sol,Talcahuano CHI,2012-06-02,"Rojas Sepulveda,E","Diaz Vasquez,M",0.5,2315.0,2212.0,A46,Queen's pawn,103.0,2263.5,1
4,1st CT Casino Marina del Sol,Talcahuano CHI,2012-06-02,"Estay Torreblanca,O","Alarcon,Rod",1.0,2214.0,2174.0,C17,French,40.0,2194.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3255687,Vladimir Dvorkovich Open,Aktobe KAZ,2026-06-08,"Kabinazar,Nurmukhammed","Tarhan,Adar",0.5,2312.0,2482.0,B11,Caro-Kann,-170.0,2397.0,0
3255688,Vladimir Dvorkovich Open,Aktobe KAZ,2026-06-08,"Zhalmakhanov,Ramazan","Kornienko,Konstantin",0.0,2478.0,2388.0,B28,Sicilian,90.0,2433.0,1
3255689,Vladimir Dvorkovich Open,Aktobe KAZ,2026-06-08,"Dronavalli,Harika","Degenbaev,Aziz",0.5,2466.0,2320.0,A06,Reti opening,146.0,2393.0,1
3255690,Vladimir Dvorkovich Open,Aktobe KAZ,2026-06-08,"Rozum,I","Makhnev,Denis",1.0,2455.0,2551.0,A40,Queen's pawn,-96.0,2503.0,0


History

In [13]:
player_stats = {}

WhiteGames = []
WhiteWin = []
WhiteLoss = []
WhiteDraw = []
BlackGames = []
BlackWin = []
BlackLoss = []
BlackDraw = []

WhiteWinRate = []
WhiteLossRate = []
WhiteDrawRate = []
BlackWinRate = []
BlackLossRate = []
BlackDrawRate = []

WLast10 = []
BLast10 = []

WasWGames = []
WasWWin = []
WasWLoss = []
WasWDraw = []
WasBGames = []
WasBWin = []
WasBLoss = []
WasBDraw = []
BasWGames = []
BasWWin = []
BasWLoss = []
BasWDraw = []
BasBGames = []
BasBWin = []
BasBLoss = []
BasBDraw = []

WasWWinRate = []
WasWLossRate = []
WasWDrawRate = []
WasBWinRate = []
WasBLossRate = []
WasBDrawRate = []
BasWWinRate = []
BasWLossRate = []
BasWDrawRate = []
BasBWinRate = []
BasBLossRate = []
BasBDrawRate = []


def update_player(player, result, color):
    stats = player_stats.get(player)
    stats["games"] += 1
    if result == 1:
        if color == "white":
            stats["win"] += 1
            stats["whitegames"] += 1
            stats["whitewin"] += 1
            stats["last10"].append(result)
        else:
            stats["loss"] += 1
            stats["blackgames"] += 1
            stats["blackloss"] += 1
            stats["last10"].append(1 - result)
    elif result == 0.5:
        if color == "white":
            stats["draw"] += 1
            stats["whitegames"] += 1
            stats["whitedraw"] += 1
            stats["last10"].append(result)
        else:
            stats["draw"] += 1
            stats["blackgames"] += 1
            stats["blackdraw"] += 1
            stats["last10"].append(1 - result)
    else:
        if color == "white":
            stats["loss"] += 1
            stats["whitegames"] += 1
            stats["whiteloss"] += 1
            stats["last10"].append(result)
        else:
            stats["win"] += 1
            stats["blackgames"] += 1
            stats["blackwin"] += 1
            stats["last10"].append(1 - result)

In [ ]:
for row in games.itertuples():
    white = row.White
    black = row.Black
    result = row.Result
    

    if white not in player_stats:
        player_stats[white] = {"games": 0, "win": 0, "loss": 0, "draw": 0, "whitegames": 0, "blackgames": 0, "whitewin": 0, "whiteloss": 0, "whitedraw": 0, "blackwin": 0, "blackloss": 0, "blackdraw": 0, "last10": deque([], maxlen=10)}
    if black not in player_stats:
        player_stats[black] = {"games": 0, "win": 0, "loss": 0, "draw": 0, "whitegames": 0, "blackgames": 0, "whitewin": 0, "whiteloss": 0, "whitedraw": 0, "blackwin": 0, "blackloss": 0, "blackdraw": 0, "last10": deque([], maxlen=10)}


    white_stats = player_stats.get(white)
    black_stats = player_stats.get(black)

    WhiteGames.append(white_stats["games"])
    WhiteWin.append(white_stats["win"])
    WhiteLoss.append(white_stats["loss"])
    WhiteDraw.append(white_stats["draw"])
    BlackGames.append(black_stats["games"])
    BlackWin.append(black_stats["win"])
    BlackLoss.append(black_stats["loss"])
    BlackDraw.append(black_stats["draw"])
    WasWGames.append(white_stats["whitegames"])
    WasWWin.append(white_stats["whitewin"])
    WasWLoss.append(white_stats["whiteloss"])
    WasWDraw.append(white_stats["whitedraw"])
    WasBGames.append(white_stats["blackgames"])
    WasBWin.append(white_stats["blackwin"])
    WasBLoss.append(white_stats["blackloss"])
    WasBDraw.append(white_stats["blackdraw"])
    BasWGames.append(black_stats["whitegames"])
    BasWWin.append(black_stats["whitewin"])
    BasWLoss.append(black_stats["whiteloss"])
    BasWDraw.append(black_stats["whitedraw"])
    BasBGames.append(black_stats["blackgames"])
    BasBWin.append(black_stats["blackwin"])
    BasBLoss.append(black_stats["blackloss"])
    BasBDraw.append(black_stats["blackdraw"])


    if white_stats["games"]:
        WhiteWinRate.append(white_stats["win"] / white_stats["games"])
        WhiteLossRate.append(white_stats["loss"] / white_stats["games"])
        WhiteDrawRate.append(white_stats["draw"] / white_stats["games"])
    else:
        WhiteWinRate.append(0)
        WhiteLossRate.append(0)
        WhiteDrawRate.append(0)
        
    if black_stats["games"]:
        BlackWinRate.append(black_stats["win"] / black_stats["games"])
        BlackLossRate.append(black_stats["loss"] / black_stats["games"])
        BlackDrawRate.append(black_stats["draw"] / black_stats["games"])
    else:
        BlackWinRate.append(0)
        BlackLossRate.append(0)
        BlackDrawRate.append(0)

    if white_stats["whitegames"]:
        WasWWinRate.append(white_stats["whitewin"] / white_stats["whitegames"])
        WasWLossRate.append(white_stats["whiteloss"] / white_stats["whitegames"])
        WasWDrawRate.append(white_stats["whitedraw"] / white_stats["whitegames"])
    else:
        WasWWinRate.append(0)
        WasWLossRate.append(0)
        WasWDrawRate.append(0)

    if white_stats["blackgames"]:
        WasBWinRate.append(white_stats["blackwin"] / white_stats["blackgames"])
        WasBLossRate.append(white_stats["blackloss"] / white_stats["blackgames"])
        WasBDrawRate.append(white_stats["blackdraw"] / white_stats["blackgames"])
    else:
        WasBWinRate.append(0)
        WasBLossRate.append(0)
        WasBDrawRate.append(0)

    if black_stats["whitegames"]:
        BasWWinRate.append(black_stats["whitewin"] / black_stats["whitegames"])
        BasWLossRate.append(black_stats["whiteloss"] / black_stats["whitegames"])
        BasWDrawRate.append(black_stats["whitedraw"] / black_stats["whitegames"])
    else:
        BasWWinRate.append(0)
        BasWLossRate.append(0)
        BasWDrawRate.append(0)

    if black_stats["blackgames"]:
        BasBWinRate.append(black_stats["blackwin"] / black_stats["blackgames"])
        BasBLossRate.append(black_stats["blackloss"] / black_stats["blackgames"])
        BasBDrawRate.append(black_stats["blackdraw"] / black_stats["blackgames"])
    else:
        BasBWinRate.append(0)
        BasBLossRate.append(0)
        BasBDrawRate.append(0)
    
    if white_stats["last10"]:
        WLast10.append(sum(white_stats["last10"]) / len(white_stats["last10"]))
    else:
        WLast10.append(0)
    if black_stats["last10"]:
        BLast10.append(sum(black_stats["last10"]) / len(black_stats["last10"]))
    else:
        BLast10.append(0)
    

    ###updating the features
    update_player(white, result, "white")
    update_player(black, result, "black")

    
    print(white_stats)
    print(black_stats)
    break

Pandas(Index=0, Event='1st CT Casino Marina del Sol', Site='Talcahuano CHI', Date='2012-06-02', White='Caceres Leal,Jaime', Black='Farfan Ortiz,V', Result=0.5, WhiteElo=2137.0, BlackElo=2206.0, ECO='D13', Opening='QGD Slav', EloDiff=-69.0, AvgElo=2171.5, HigherRatedPlayer=0)
{'games': 1, 'win': 0, 'loss': 0, 'draw': 1, 'whitegames': 1, 'blackgames': 0, 'whitewin': 0, 'whiteloss': 0, 'whitedraw': 1, 'blackwin': 0, 'blackloss': 0, 'blackdraw': 0, 'last10': deque([0.5], maxlen=10)}
{'games': 1, 'win': 0, 'loss': 0, 'draw': 1, 'whitegames': 0, 'blackgames': 1, 'whitewin': 0, 'whiteloss': 0, 'whitedraw': 0, 'blackwin': 0, 'blackloss': 0, 'blackdraw': 1, 'last10': deque([0.5], maxlen=10)}
